# IDX Exchange DS Internship - Week Four -  Baseline Model

* Train a Linear Regression as the first model.
* Evaluate using R² on the test set.
* Record baseline results.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error

from sklearn.preprocessing import TargetEncoder

## Import Data

In [2]:
# Read in the training and test sets created last week

train_and_val = pd.read_csv("week_3_train.csv")
testing = pd.read_csv("week_3_test.csv")

In [30]:
# Create a validation set out of the training data

# Training Set: January 2024 - February 2026
# Validation Set: March 2026
# Test Set: April 2026

training = train_and_val[~(train_and_val["PropertyMonth"] == "2026-03-01")]
validation = train_and_val[train_and_val["PropertyMonth"] == "2026-03-01"]

print(train_and_val.shape, training.shape, validation.shape)

(265391, 54) (255097, 54) (10294, 54)


In [31]:
training.columns

Index(['ViewYN', 'PoolPrivateYN', 'ListingKey', 'ClosePrice', 'Latitude',
       'Longitude', 'LivingArea', 'ListOfficeName', 'CountyOrParish',
       'AttachedGarageYN', 'ParkingTotal', 'YearBuilt', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'FireplaceYN',
       'Stories', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces',
       'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'PropertyMonth', 'ClosePriceLog', 'LivingAreaLog',
       'LotSizeSquareFeetLog', 'AssociationFeeLog', 'Bamboo', 'Brick',
       'Carpet', 'Concrete', 'Laminate', 'SeeRemarks', 'Stone', 'Tile',
       'Unknown', 'Vinyl', 'Wood', 'StateOrProvince_AZ', 'StateOrProvince_CA',
       'StateOrProvince_CO', 'StateOrProvince_ME', 'StateOrProvince_NV',
       'StateOrProvince_OS', 'StateOrProvince_TN', 'ListOfficeName_enc',
       'CountyOrParish_enc', 'HighSchoolDistrict_enc', 'City_enc',
       'PostalCode_enc', 'YearBuilt_enc'],
      dtype='object')

In [2]:
def median_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.median(np.abs((y_true - y_pred) / y_true))

def metrics(y_true, y_pred):
    # Note: you have to undeo the log transform on ClosePrice by using np.exp()!
    y_true = np.exp(y_true)
    y_pred = np.exp(y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    mdape = median_absolute_percentage_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    return r2, mape * 100, mdape * 100, mae

def metrics_no_log(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    mdape = median_absolute_percentage_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    return r2, mape * 100, mdape * 100, mae

def metric_df(r2_train, mape_train, mdape_train, mae_train,
              r2_val, mape_val, mdape_val, mae_val,
              r2_test, mape_test, mdape_test, mae_test):
    data = {"r2": [r2_train, r2_val, r2_test],
            "MAPE": [mape_train, mape_val, mape_test],
            "MdAPE": [mdape_train, mdape_val, mdape_test],
            "MAE": [f"${round(mae_train, 2)}", f"${round(mae_val, 2)}", f"${round(mae_test, 2)}"]}
    df = pd.DataFrame(data)
    df.index = ["Train Set", "Validation Set", "Test Set"]
    return df

In [33]:
# Columns not being used: "ListingKey", "ListingId", "ClosePrice", "ListOfficeName", "CountyOrParish", "YearBuilt",
#         "City", "HighSchoolDistrict", "PostalCode", "LivingArea", "LotSizeSquareFeet", "AssociationFee", "StateOrProvince_CO",
#         "StateOrProvince_ME", "StateOrProvince_NV", "StateOrProvince_NV", "StateOrProvince_OS", "StateOrProvince_TN"

common_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "StateOrProvince_AZ", "StateOrProvince_CA",
            "ListOfficeName_enc", "CountyOrParish_enc", "HighSchoolDistrict_enc", "City_enc", "PostalCode_enc", "YearBuilt_enc",
            "PropertyMonth", "ClosePriceLog"]

# The x_train window will be set per model, so don't define it here

x_val = validation[["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "StateOrProvince_AZ", "StateOrProvince_CA",
            "ListOfficeName_enc", "CountyOrParish_enc", "HighSchoolDistrict_enc", "City_enc", "PostalCode_enc", "YearBuilt_enc"]]  
y_val = validation["ClosePriceLog"]


x_test = testing[["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "StateOrProvince_AZ", "StateOrProvince_CA",
            "ListOfficeName_enc", "CountyOrParish_enc", "HighSchoolDistrict_enc", "City_enc", "PostalCode_enc", "YearBuilt_enc"]]
y_test = testing["ClosePriceLog"]

## Model One: Baseline

### Train Set: January 2024 - February 2026

In [34]:
# Use the full set of available months (26 months)

x_train = training[common_cols].drop(columns = ["PropertyMonth", "ClosePriceLog"])
y_train = training["ClosePriceLog"]

In [35]:
model = LinearRegression()
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test)

In [36]:
model_one_v1 = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_one_v1

,r2,MAPE,MdAPE,MAE
Train Set,0.833892,13.688969,10.594696,$153636.91
Validation Set,0.822246,14.140095,11.067966,$160440.11
Test Set,0.845126,13.320288,10.430795,$156034.55


### Train Set: February 2025 - February 2026

In [10]:
# Shorten the training window to the year preceding the validation set (12 months)

past_limit = pd.to_datetime("2025-01-01")
present_limit = pd.to_datetime("2026-02-01")

x_train_with_month = training[common_cols]
x_train_with_month.loc[:, "PropertyMonth"] = pd.to_datetime(x_train_with_month["PropertyMonth"])
x_train = x_train_with_month[(x_train_with_month["PropertyMonth"] >= past_limit) & (x_train_with_month["PropertyMonth"] <= present_limit)]
y_train = x_train["ClosePriceLog"]
x_train = x_train.drop(columns = ["PropertyMonth", "ClosePriceLog"])

In [11]:
model = LinearRegression()
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test)

In [12]:
model_one_v2 = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_one_v2

,r2,MAPE,MdAPE,MAE
Train Set,0.827295,13.887947,10.764396,$154953.64
Validation Set,0.822232,14.142001,11.097116,$160497.2
Test Set,0.844340,13.362200,10.457773,$156581.48


### Train Set: July 2024 - February 2026

In [13]:
# Set the training window to 20 months preceding the validation set

past_limit = pd.to_datetime("2024-06-01")
present_limit = pd.to_datetime("2026-02-01")

x_train_with_month = training[common_cols]
x_train_with_month.loc[:, "PropertyMonth"] = pd.to_datetime(x_train_with_month["PropertyMonth"])
x_train = x_train_with_month[(x_train_with_month["PropertyMonth"] >= past_limit) & (x_train_with_month["PropertyMonth"] <= present_limit)]
y_train = x_train["ClosePriceLog"]
x_train = x_train.drop(columns = ["PropertyMonth", "ClosePriceLog"])

In [14]:
model = LinearRegression()
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test)

In [15]:
model_one_v3 = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_one_v3

,r2,MAPE,MdAPE,MAE
Train Set,0.833079,13.680969,10.585634,$153605.05
Validation Set,0.822262,14.154090,11.104972,$160490.74
Test Set,0.844849,13.351712,10.423243,$156274.5


## Model Two: No Log Transform on Target

In [16]:
to_encode_train = train_and_val[["ListOfficeName", "CountyOrParish", "HighSchoolDistrict", "City", "PostalCode", "YearBuilt"]]
to_encode_test = testing[["ListOfficeName", "CountyOrParish", "HighSchoolDistrict", "City", "PostalCode", "YearBuilt"]]

y_train = train_and_val["ClosePrice"]

In [17]:
encoder = TargetEncoder(smooth="auto", cv=5, random_state=42)

x_train_and_val_encoded = encoder.fit_transform(to_encode_train, y_train)
x_test_encoded = encoder.transform(to_encode_test)

x_train_and_val_encoded_df = pd.DataFrame(x_train_and_val_encoded, columns=to_encode_train.columns)
x_test_encoded_df = pd.DataFrame(x_test_encoded, columns=to_encode_train.columns)

In [18]:
new_train_val = train_and_val.join(x_train_and_val_encoded_df, rsuffix="_no_log_enc")

new_training = new_train_val[~(new_train_val["PropertyMonth"] == "2026-03-01")]
new_validation = new_train_val[new_train_val["PropertyMonth"] == "2026-03-01"]

In [19]:
new_validation.columns

Index(['ViewYN', 'PoolPrivateYN', 'ListingKey', 'ClosePrice', 'Latitude',
       'Longitude', 'LivingArea', 'ListOfficeName', 'CountyOrParish',
       'AttachedGarageYN', 'ParkingTotal', 'YearBuilt', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'FireplaceYN',
       'Stories', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces',
       'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'PropertyMonth', 'ClosePriceLog', 'LivingAreaLog',
       'LotSizeSquareFeetLog', 'AssociationFeeLog', 'Bamboo', 'Brick',
       'Carpet', 'Concrete', 'Laminate', 'SeeRemarks', 'Stone', 'Tile',
       'Unknown', 'Vinyl', 'Wood', 'StateOrProvince_AZ', 'StateOrProvince_CA',
       'StateOrProvince_CO', 'StateOrProvince_ME', 'StateOrProvince_NV',
       'StateOrProvince_OS', 'StateOrProvince_TN', 'ListOfficeName_enc',
       'CountyOrParish_enc', 'HighSchoolDistrict_enc', 'City_enc',
       'PostalCode_enc', 'YearBuilt_enc', 'ListOfficeName_no_log

In [20]:
common_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "StateOrProvince_AZ", "StateOrProvince_CA",
            "ListOfficeName_no_log_enc", "CountyOrParish_no_log_enc", "HighSchoolDistrict_no_log_enc",
            "City_no_log_enc", "PostalCode_no_log_enc", "YearBuilt_no_log_enc",
            "PropertyMonth", "ClosePriceLog"]

In [21]:
x_train = new_training[common_cols].drop(columns = ["PropertyMonth", "ClosePriceLog"])
y_train = new_training["ClosePrice"]

x_val = new_validation[["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "StateOrProvince_AZ", "StateOrProvince_CA",
            "ListOfficeName_no_log_enc", "CountyOrParish_no_log_enc", "HighSchoolDistrict_no_log_enc",
            "City_no_log_enc", "PostalCode_no_log_enc", "YearBuilt_no_log_enc"]]  
y_val = new_validation["ClosePrice"]

x_test = testing.join(x_test_encoded_df, rsuffix="_no_log_enc")
x_test = x_test[["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "StateOrProvince_AZ", "StateOrProvince_CA",
             "ListOfficeName_no_log_enc", "CountyOrParish_no_log_enc", "HighSchoolDistrict_no_log_enc",
            "City_no_log_enc", "PostalCode_no_log_enc", "YearBuilt_no_log_enc"]]  
y_test = testing["ClosePrice"]

In [22]:
model = LinearRegression()
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics_no_log(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics_no_log(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics_no_log(y_test, pred_test)

In [23]:
model_two = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_two

,r2,MAPE,MdAPE,MAE
Train Set,0.811542,18.254361,14.134041,$180588.97
Validation Set,0.802439,18.573902,14.455737,$186124.71
Test Set,0.825990,17.631408,13.524996,$181177.6


## Model Three: BedBathRatio and PropertyAge

In [3]:
# Read in the training and test sets created in week six

train_and_val = pd.read_csv("feat_eng_train.csv")
testing = pd.read_csv("feat_eng_test.csv")

In [4]:
training = train_and_val[~(train_and_val["PropertyMonth"] == "2026-03-01")]
validation = train_and_val[train_and_val["PropertyMonth"] == "2026-03-01"]

print(train_and_val.shape, training.shape, validation.shape)

(265314, 163) (255022, 163) (10292, 163)


In [5]:
common_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "StateOrProvince_AZ", "StateOrProvince_CA",
            "ListOfficeName_enc", "CountyOrParish_enc", "HighSchoolDistrict_enc", "City_enc", "PostalCode_enc", "YearBuilt_enc",
            "BedBathRatio", "PropertyAge"]

x_train = training[common_cols]
y_train = training["ClosePriceLog"]

x_val = validation[common_cols]  
y_val = validation["ClosePriceLog"]


x_test = testing[common_cols]
y_test = testing["ClosePriceLog"]

In [6]:
model = LinearRegression()
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test)

In [7]:
model_three = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_three

,r2,MAPE,MdAPE,MAE
Train Set,0.836278,13.529778,10.429609,$152288.02
Validation Set,0.825397,13.935469,10.892233,$158695.59
Test Set,0.846540,13.149937,10.305918,$154923.08


## Model Four: Improved School Districts

### Both Counts and Percents

In [10]:
# During the week six feature engineering, the MLS data was joined with a California School District Area dataset
# That dataset contains school district names, as well as demographic data within that district
# This section will include both the percent and count columns for demographic information

common_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "StateOrProvince_AZ", "StateOrProvince_CA",
            "ListOfficeName_enc", "CountyOrParish_enc", "City_enc", "PostalCode_enc", "YearBuilt_enc",
            "BedBathRatio", "PropertyAge",
            "DistrictName_elem_enc", "DistrictName_unif_enc", "DistrictName_high_enc",
            'EnrollTotal_high', 'EnrollCharter_high', 'EnrollNonCharter_high', 'AApct_high', 'AAcount_high', 'AIpct_high',
            'AIcount_high', 'ASpct_high', 'AScount_high', 'FIpct_high', 'FIcount_high', 'HIpct_high', 'HIcount_high', 'PIpct_high',
            'PIcount_high', 'WHpct_high', 'WHcount_high', 'MRpct_high', 'MRcount_high', 'NRpct_high', 'NRcount_high', 'ELpct_high',
            'ELcount_high', 'FOSpct_high', 'FOScount_high','HOMpct_high','HOMcount_high','MIGpct_high','MIGcount_high','SWDpct_high',
            'SWDcount_high','SEDpct_high','SEDcount_high', 'EnrollTotal_elem','EnrollCharter_elem','EnrollNonCharter_elem','AApct_elem',
            'AAcount_elem','AIpct_elem','AIcount_elem','ASpct_elem','AScount_elem','FIpct_elem','FIcount_elem','HIpct_elem','HIcount_elem',
            'PIpct_elem','PIcount_elem','WHpct_elem','WHcount_elem','MRpct_elem','MRcount_elem','NRpct_elem','NRcount_elem','ELpct_elem',
            'ELcount_elem','FOSpct_elem','FOScount_elem','HOMpct_elem','HOMcount_elem','MIGpct_elem','MIGcount_elem','SWDpct_elem',
            'SWDcount_elem','SEDpct_elem','SEDcount_elem','EnrollTotal_unif','EnrollCharter_unif','EnrollNonCharter_unif','AApct_unif',
            'AAcount_unif','AIpct_unif','AIcount_unif','ASpct_unif','AScount_unif','FIpct_unif','FIcount_unif','HIpct_unif','HIcount_unif',
            'PIpct_unif','PIcount_unif','WHpct_unif','WHcount_unif','MRpct_unif','MRcount_unif','NRpct_unif','NRcount_unif','ELpct_unif',
            'ELcount_unif','FOSpct_unif','FOScount_unif','HOMpct_unif','HOMcount_unif','MIGpct_unif','MIGcount_unif','SWDpct_unif',
            'SWDcount_unif','SEDpct_unif','SEDcount_unif']

x_train = training[common_cols]
y_train = training["ClosePriceLog"]

x_val = validation[common_cols]  
y_val = validation["ClosePriceLog"]


x_test = testing[common_cols]
y_test = testing["ClosePriceLog"]

In [11]:
model = LinearRegression()
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test)

In [12]:
model_four_v1 = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_four_v1

,r2,MAPE,MdAPE,MAE
Train Set,0.844151,12.919246,9.771261,$146208.83
Validation Set,0.832746,13.314274,10.198476,$152792.45
Test Set,0.855694,12.551435,9.564880,$148433.15


### Only Percents

In [14]:
# Test if model performance changes when only the percent columns are used for demographic information

common_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "StateOrProvince_AZ", "StateOrProvince_CA",
            "ListOfficeName_enc", "CountyOrParish_enc", "City_enc", "PostalCode_enc", "YearBuilt_enc",
            "BedBathRatio", "PropertyAge",
            "DistrictName_elem_enc", "DistrictName_unif_enc", "DistrictName_high_enc",
            'EnrollTotal_high', 'EnrollCharter_high', 'EnrollNonCharter_high', 'AApct_high', 'AIpct_high',
            'ASpct_high','FIpct_high','HIpct_high','PIpct_high',
            'WHpct_high','MRpct_high', 'NRpct_high', 'ELpct_high',
            'FOSpct_high', 'HOMpct_high','MIGpct_high','SWDpct_high',
            'SEDpct_high','EnrollTotal_elem','EnrollCharter_elem','EnrollNonCharter_elem','AApct_elem',
            'AIpct_elem','ASpct_elem','FIpct_elem','HIpct_elem',
            'PIpct_elem','WHpct_elem','MRpct_elem','NRpct_elem','ELpct_elem',
            'FOSpct_elem','HOMpct_elem','MIGpct_elem','SWDpct_elem',
            'SEDpct_elem','EnrollTotal_unif','EnrollCharter_unif','EnrollNonCharter_unif','AApct_unif',
            'AIpct_unif','ASpct_unif','FIpct_unif','HIpct_unif',
            'PIpct_unif','WHpct_unif','MRpct_unif','NRpct_unif','ELpct_unif',
            'FOSpct_unif','HOMpct_unif','MIGpct_unif','SWDpct_unif',
            'SEDpct_unif']

x_train = training[common_cols]
y_train = training["ClosePriceLog"]

x_val = validation[common_cols]  
y_val = validation["ClosePriceLog"]

x_test = testing[common_cols]
y_test = testing["ClosePriceLog"]

In [15]:
model = LinearRegression()
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test)

In [16]:
model_four_v2 = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_four_v2

,r2,MAPE,MdAPE,MAE
Train Set,0.842304,13.084040,9.978489,$147743.57
Validation Set,0.830719,13.470752,10.510910,$154240.36
Test Set,0.854175,12.696367,9.750873,$149772.78


### Only Counts

In [20]:
# Test if model performance changes when only the count columns are used for demographic information

common_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "StateOrProvince_AZ", "StateOrProvince_CA",
            "ListOfficeName_enc", "CountyOrParish_enc", "City_enc", "PostalCode_enc", "YearBuilt_enc",
            "BedBathRatio", "PropertyAge",
            "DistrictName_elem_enc", "DistrictName_unif_enc", "DistrictName_high_enc",
            'EnrollTotal_high', 'EnrollCharter_high', 'EnrollNonCharter_high', 'AAcount_high', 'AIcount_high',
            'AScount_high','FIcount_high','HIcount_high','PIcount_high',
            'WHcount_high','MRcount_high', 'NRcount_high', 'ELcount_high',
            'FOScount_high', 'HOMcount_high','MIGcount_high','SWDcount_high',
            'SEDcount_high','EnrollTotal_elem','EnrollCharter_elem','EnrollNonCharter_elem','AAcount_elem',
            'AIcount_elem','AScount_elem','FIcount_elem','HIcount_elem',
            'PIcount_elem','WHcount_elem','MRcount_elem','NRcount_elem','ELcount_elem',
            'FOScount_elem','HOMcount_elem','MIGcount_elem','SWDcount_elem',
            'SEDcount_elem','EnrollTotal_unif','EnrollCharter_unif','EnrollNonCharter_unif','AAcount_unif',
            'AIcount_unif','AScount_unif','FIcount_unif','HIcount_unif',
            'PIcount_unif','WHcount_unif','MRcount_unif','NRcount_unif','ELcount_unif',
            'FOScount_unif','HOMcount_unif','MIGcount_unif','SWDcount_unif',
            'SEDcount_unif']

x_train = training[common_cols]
y_train = training["ClosePriceLog"]

x_val = validation[common_cols]  
y_val = validation["ClosePriceLog"]

x_test = testing[common_cols]
y_test = testing["ClosePriceLog"]

In [21]:
model = LinearRegression()
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test)

In [22]:
model_four_v3 = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_four_v3

,r2,MAPE,MdAPE,MAE
Train Set,0.841861,13.072304,9.948106,$147861.19
Validation Set,0.831310,13.422834,10.288210,$153953.46
Test Set,0.853599,12.670559,9.659332,$149901.75


# Summary

This table presents the best performing model from each section.

| Model | Test r2 | Test MAPE | Test MdAPE | Test MAE | 
| ----- | ------- | --------- | ---------- | -------- |
| Model One (Baseline) | 0.845126 | 13.320288 | 10.430795 | \$156,581|
| Model Two (Removing Log Transform on Target Variable) | 0.825990 | 17.631408 | 13.524996 | \$181,177|
| Model Three (Feature Engineering Additions) | 0.846540 | 13.149937 | 10.305918 | \$154,923|
| **Model Four (School District Information)** | **0.855694** | **12.551435** | **9.564880** | **\$148,433**|